# Few-Shot Domain Calibration ke Trafik Nyata AWS

**Dijalankan di SageMaker.** Menutup lingkaran validasi lintas-jaringan: model NIDS
yang gagal *zero-shot* di AWS (Fase 2: MCC~0) di-*repair* dengan **few-shot** memakai
sedikit label AWS. Membuktikan tesis: **SFM + few-shot** = solusi adaptasi ke ruang
(lingkungan) baru yang murah dan diulang tiap lingkungan.

**Rancangan (anti-bocor):**
1. Rakit dataset AWS berlabel 9-fitur (dari `detect_clean_flows.csv` + `detect_volumetric_flows.csv` di S3).
2. Split **stratified** AWS -> `aws_train` (kolam few-shot) & `aws_test` (uji tetap, tak pernah dilatih).
3. **Baseline zero-shot**: model source (UNS/CIC) diuji langsung ke `aws_test` -> MCC (harusnya ~0).
4. **Few-shot**: latih XGBoost dari `source + k% aws_train` (k=1,5,10,20,50%), uji `aws_test`.
5. **AWS-only** (acuan atas): latih hanya dari `aws_train`, uji `aws_test`.
6. Simpan `aws_fewshot_results.json` + kurva PNG, **UPLOAD** ke S3 `unsw-far/fewshot/`.

> Source (CIC/UNS 9-fitur) diambil dari `*_flows_v2.csv` hasil notebook 21 bila ada,
> atau dibangun ulang dari data mentah (pkl/csv) dengan pemetaan & satuan yang sama.

In [ ]:
import importlib, sys, subprocess
pkgmap={'sklearn':'scikit-learn'}
need=[m for m in ('xgboost','sklearn','pandas','numpy','matplotlib','boto3') if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*[pkgmap.get(m,m) for m in need]],check=True)
print('setup ok' if not need else f'installed {need}')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score
import xgboost as xgb
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3})

S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='fewshot_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
SEED=42
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','seed':SEED,'features':CANON}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def metrics(y,yp): return dict(mcc=round(float(matthews_corrcoef(y,yp)),4),f1=round(float(f1_score(y,yp,zero_division=0)),4),precision=round(float(precision_score(y,yp,zero_division=0)),4),recall=round(float(recall_score(y,yp,zero_division=0)),4),n=int(len(y)),n_pos=int(np.sum(y)))

## 1. Rakit dataset AWS berlabel (dari S3)

In [ ]:
AWS_DIR='aws_labeled'; os.makedirs(AWS_DIR,exist_ok=True)
want=['detect_clean_flows.csv','detect_volumetric_flows.csv']
try:
    import boto3; s3=boto3.client('s3',region_name=REGION)
    for fn in want:
        dst=os.path.join(AWS_DIR,fn)
        if not os.path.exists(dst): s3.download_file(S3_BUCKET,f'{S3_PREFIX}/results/{fn}',dst)
    print('unduh label AWS ok')
except Exception as e: print('download S3 dilewati:',e)

parts=[]
for fn in want:
    p=os.path.join(AWS_DIR,fn)
    if os.path.exists(p):
        d=pd.read_csv(p)
        if 'ground_truth' in d.columns and all(c in d.columns for c in CANON):
            d=d[CANON+['ground_truth']].rename(columns={'ground_truth':'y'}); d['src_file']=fn; parts.append(d)
aws=pd.concat(parts,ignore_index=True).replace([np.inf,-np.inf],np.nan).dropna()
aws['y']=aws['y'].astype(int)
print('AWS berlabel:',len(aws),'| komposisi:',aws['y'].value_counts().to_dict())
RESULTS['aws_total']={'n':int(len(aws)),'n_attack':int(aws['y'].sum()),'n_benign':int((aws['y']==0).sum())}

## 2. Split stratified train/test AWS (anti-bocor)
`aws_test` tetap sepanjang eksperimen. Karena benign AWS sedikit, kita seimbangkan
kolam few-shot dengan *undersample* attack agar rasio tak ekstrem (opsional).

In [ ]:
from sklearn.model_selection import train_test_split
aws_tr, aws_te = train_test_split(aws, test_size=0.4, stratify=aws['y'], random_state=SEED)
print('train:',len(aws_tr),aws_tr['y'].value_counts().to_dict(),'| test:',len(aws_te),aws_te['y'].value_counts().to_dict())
RESULTS['split']={'train':int(len(aws_tr)),'test':int(len(aws_te)),
                  'test_pos':int(aws_te['y'].sum()),'test_neg':int((aws_te['y']==0).sum())}
Xte, yte = aws_te[CANON].values, aws_te['y'].values

## 3. Muat SOURCE (UNS/CIC 9-fitur) untuk few-shot & baseline

In [ ]:
# SOURCE benign+attack berlabel diperlukan untuk melatih model source.
# UNSW punya label (kolom 'label'); CIC pkl punya 'y'. Kita bangun keduanya bila tersedia.
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None

def load_uns():
    f=first(['../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])
    if not f: return None
    u=pd.read_csv(f); need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','label']
    if any(c not in u.columns for c in need): return None
    X=pd.DataFrame({'duration':pd.to_numeric(u['dur'],errors='coerce')*1e6,'fwd_pkts':u['spkts'],'bwd_pkts':u['dpkts'],
                    'fwd_bytes':u['sbytes'],'bwd_bytes':u['dbytes'],'fwd_mean':u['smean'],'bwd_mean':u['dmean'],
                    'src_load':pd.to_numeric(u['sload'],errors='coerce')/8.0,
                    'dst_load':pd.to_numeric(u['dpkts'],errors='coerce')/pd.to_numeric(u['dur'],errors='coerce').replace(0,np.nan)})
    X['y']=u['label'].astype(int); X=X.replace([np.inf,-np.inf],np.nan).dropna()
    return X[CANON+['y']]

def load_cic():
    f=first(['../../CICDDoS2018/data/cleaned_100.pkl','../../CICDDoS2018/data/cleaned_*.pkl'])
    if not f: return None
    import pickle
    with open(f,'rb') as fh: d=pickle.load(fh)
    feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
    Xo=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
    cdf=pd.DataFrame(Xo,columns=feats)
    cmap={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
          'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    if any(v not in cdf.columns for v in cmap.values()): return None
    out=pd.DataFrame({k:cdf[cmap[k]].values for k in CANON})
    out['y']=np.asarray(d['y']).astype(int) if 'y' in d else np.asarray(d.get('labels')).astype(int)
    return out[CANON+['y']].replace([np.inf,-np.inf],np.nan).dropna()

SOURCES={}
for name,loader in [('UNS',load_uns),('CIC',load_cic)]:
    s=loader()
    if s is not None and len(s)>0:
        SOURCES[name]=s; print(f'SOURCE {name}: n={len(s)} komposisi={s["y"].value_counts().to_dict()}')
    else:
        print(f'SOURCE {name}: TIDAK tersedia (dilewati)')
RESULTS['sources_available']=list(SOURCES.keys())
assert SOURCES, 'Tidak ada source (UNS/CIC) yang bisa dimuat -- cek path data mentah.'

## 4. Baseline ZERO-SHOT (source-only) -> uji AWS-test

In [ ]:
def train_xgb(X,y,seed=SEED):
    y=np.asarray(y); n_pos=int((y==1).sum()); n_neg=int((y==0).sum())
    params=dict(max_depth=8,learning_rate=0.1,n_estimators=300,subsample=0.9,
                colsample_bytree=0.9,eval_metric='logloss',n_jobs=-1,random_state=seed)
    # scale_pos_weight hanya bermakna bila kedua kelas ada (hindari warning 'not used')
    if n_pos>0 and n_neg>0: params['scale_pos_weight']=n_neg/n_pos
    clf=xgb.XGBClassifier(**params); clf.fit(X,y); return clf

def sample_frac(df,frac,rng):
    if frac>=1.0: return df
    outs=[]
    for c,g in df.groupby('y'):
        n=max(1,int(round(len(g)*frac))); outs.append(g.iloc[rng.permutation(len(g))[:n]])
    return pd.concat(outs)

def run_source(name, src):
    rng=np.random.default_rng(SEED); rows=[]
    # zero-shot: source-only -> AWS-test
    m=train_xgb(src[CANON].values, src['y'].values)
    r=metrics(yte,m.predict(Xte)); r.update(source=name,k_percent=0.0,mode=f'zero-shot {name}')
    rows.append(r); print(f'[{name}] zero-shot -> MCC={r["mcc"]} F1={r["f1"]} recall={r["recall"]}')
    # few-shot: source + k% AWS
    for k in [1,5,10,20,50]:
        aws_k=sample_frac(aws_tr,k/100.0,rng)
        Xtr=np.vstack([src[CANON].values, aws_k[CANON].values]); ytr=np.r_[src['y'].values, aws_k['y'].values]
        m=train_xgb(Xtr,ytr); r=metrics(yte,m.predict(Xte))
        r.update(source=name,k_percent=float(k),mode=f'few-shot {name}+{k}%AWS',n_aws_train=int(len(aws_k)))
        rows.append(r); print(f'[{name}] few-shot {k}% (aws n={len(aws_k)}) -> MCC={r["mcc"]} recall={r["recall"]}')
    return rows

## 5. FEW-SHOT: source + k% AWS-train

In [ ]:
res_rows=[]
for name,src in SOURCES.items():
    res_rows += run_source(name, src)

## 6. Acuan atas: AWS-only (latih aws_train, uji aws_test)

In [ ]:
m_aws=train_xgb(aws_tr[CANON].values, aws_tr['y'].values)
r=metrics(yte,m_aws.predict(Xte)); r.update(source='AWS',k_percent=100.0, mode='AWS-only (acuan atas)')
res_rows.append(r); print('AWS-only:',{kk:r[kk] for kk in ['mcc','f1','recall','precision']})
RESULTS['runs']=res_rows
tab=pd.DataFrame(res_rows)[['source','mode','k_percent','mcc','f1','precision','recall','n_pos','n']]
import IPython.display as ipd; ipd.display(tab)

## 7. Kurva few-shot & simpan + UPLOAD S3

In [ ]:
fig,ax=plt.subplots(figsize=(7.4,4.4))
colmap={'UNS':'#DD8452','CIC':'#4C72B0'}
for name in RESULTS.get('sources_available',[]):
    pts=[r for r in res_rows if r.get('source')==name and (r['k_percent']==0.0 or r['mode'].startswith('few-shot'))]
    pts=sorted(pts,key=lambda r:r['k_percent'])
    xs=[r['k_percent'] for r in pts]; ys=[r['mcc'] for r in pts]
    ax.plot(xs,ys,'o-',lw=2,color=colmap.get(name),label=f'{name}+AWS')
    for x,y in zip(xs,ys): ax.text(x,y+0.015,f'{y:.2f}',ha='center',fontsize=7,color=colmap.get(name))
aws_only=next((r['mcc'] for r in res_rows if r['mode'].startswith('AWS-only')),None)
if aws_only is not None: ax.axhline(aws_only,ls='--',color='#55A868',label=f'AWS-only={aws_only:.3f}')
ax.set_xlabel('% label AWS pada train'); ax.set_ylabel('MCC di AWS-test'); ax.set_ylim(-0.05,1.0)
ax.set_title('Few-shot domain calibration ke trafik nyata AWS'); ax.legend()
plt.tight_layout(); savefig('aws_fewshot_curve.png')

json_path=os.path.join(OUTDIR,'aws_fewshot_results.json')
with open(json_path,'w') as f: json.dump(RESULTS,f,indent=2)
print('tersimpan',json_path)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/fewshot/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/fewshot/')
    for o in s3.list_objects_v2(Bucket=S3_BUCKET,Prefix=f'{S3_PREFIX}/fewshot/').get('Contents',[]): print('  ',o['Key'],o['Size'],'B')
except Exception as e: print('upload gagal:',e)
print('\nSELESAI. Beri tahu asisten -> unduh s3://%s/%s/fewshot/ untuk analisis.'%(S3_BUCKET,S3_PREFIX))